In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.optim import Adam
from torch.utils.data import DataLoader
from source.version5.data import trainLoader
from source.version5.model import EfficientModel
from source.version5.train import trainModel
from source.version5.loss import CELoss
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts as CosLR
from cutmix.cutmix import CutMix
from cutmix.utils import CutMixCrossEntropyLoss

In [3]:
def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(2017)

In [4]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/raw/train_images/'
    loader['label_path'] = '../../data/raw/data.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    train = CutMix(train, num_class=5, beta=1.0, prob=0.5, num_mix=2)
    valid = CutMix(valid, num_class=5, beta=1.0, prob=0.0, num_mix=2)
    params = {}
    params['batch_size'] = 7
    params['num_workers'] = 3
    params['drop_last'] = True
    train = DataLoader(train, **params, shuffle=True)
    valid = DataLoader(valid, **params, shuffle=False)
    model = EfficientModel()
    model = model.to('cuda:0')
    optimizer = Adam(model.parameters(), lr=1e-04, weight_decay=1e-6)
    schedular = CosLR(optimizer, T_0=10, T_mult=1, eta_min=1e-6, last_epoch=-1)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = CELoss()
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version5/model_{}.pt'.format(fold)
    trainer['epochs'] = 10
    trainer['batch'] = 7
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [5]:
train(0)

Train Images: 17117 Valid Images: 4280


100% 17115/17115 [13:52<00:00, 20.56it/s, trn_ls=0.7943, val_ls=0.3377, val_mt=0.8534]
100% 17115/17115 [13:44<00:00, 20.77it/s, trn_ls=0.6496, val_ls=0.2874, val_mt=0.8716]
100% 17115/17115 [13:42<00:00, 20.81it/s, trn_ls=0.6218, val_ls=0.2628, val_mt=0.8751]
100% 17115/17115 [13:58<00:00, 20.40it/s, trn_ls=0.5858, val_ls=0.2760, val_mt=0.8735]
100% 17115/17115 [13:57<00:00, 20.44it/s, trn_ls=0.5775, val_ls=0.3095, val_mt=0.8585]
100% 17115/17115 [13:55<00:00, 20.48it/s, trn_ls=0.5528, val_ls=0.2517, val_mt=0.8845]
100% 17115/17115 [13:56<00:00, 20.47it/s, trn_ls=0.5417, val_ls=0.2543, val_mt=0.8850]
100% 17115/17115 [13:57<00:00, 20.44it/s, trn_ls=0.5373, val_ls=0.2411, val_mt=0.8880]
100% 17115/17115 [13:56<00:00, 20.46it/s, trn_ls=0.5384, val_ls=0.2485, val_mt=0.8866]
100% 17115/17115 [13:56<00:00, 20.46it/s, trn_ls=0.5313, val_ls=0.2638, val_mt=0.8826]


In [ ]:
train(1)

Train Images: 17117 Valid Images: 4280


100% 17115/17115 [13:56<00:00, 20.46it/s, trn_ls=0.7929, val_ls=0.3007, val_mt=0.8665]
100% 17115/17115 [13:58<00:00, 20.42it/s, trn_ls=0.6552, val_ls=0.2715, val_mt=0.8740]
100% 17115/17115 [14:02<00:00, 20.31it/s, trn_ls=0.6175, val_ls=0.2716, val_mt=0.8780]
100% 17115/17115 [13:54<00:00, 20.51it/s, trn_ls=0.5953, val_ls=0.2599, val_mt=0.8861]
100% 17115/17115 [13:59<00:00, 20.39it/s, trn_ls=0.5720, val_ls=0.2451, val_mt=0.8875]
 80% 13615/17115 [10:27<02:42, 21.60it/s, trn_ls=0.57310]

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)